This notebook extends the ["AI Agent (Part 1)"](./AI%20Agent%20%28Part%201%29.ipynb) example.

In [ ]:
# Install LangChain and OpenAI integration
!pip install -q langchain langchain-openai

In [ ]:
from IPython.display import Image
from google.colab import userdata              # Accesses secrets stored in Google Colab
from langchain.agents import AgentState        # Built-in state type with a 'messages' list
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.runnables import Runnable
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode
from pathlib import Path
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

# Helper: pretty-prints each message in the conversation history
def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

# Helper: renders the compiled graph as a PNG and shows it inline in Jupyter
def display_graph(runnable: Runnable, output_png: Path):
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

In [ ]:
# Same tools as Part 1 — @tool turns Python functions into LangChain tools the LLM can call.
# The docstring is what the model reads to understand when and how to use each tool.

@tool
def weather(city: str) -> str:
    """Return a (fake) current-weather report for a city."""
    data = {
        "sofia": "Sofia: 18 C, partly cloudy",
        "london": "London: 11 C, rainy",
        "tokyo": "Tokyo: 22 C, sunny",
    }
    return data.get(city.lower(), f"No data for {city}.")

@tool
def search(query: str) -> str:
    """Look up a term in the built-in mini-encyclopedia."""
    data = {
        "langgraph": "LangGraph is a library for building stateful, cyclic LLM apps.",
        "react": "ReAct is a prompting pattern: Reason then Act, in a loop.",
        "dag": "A DAG is a directed acyclic graph — no cycles allowed.",
    }
    return data.get(query.lower(), "(nothing found)")


SYSTEM_PROMPT = "You are a concise assistant. Use tools when useful."
TOOLS = [weather, search]

In [ ]:
# Bind tools to the GPT model (same as Part 1)
model = ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key).bind_tools(TOOLS)

# NEW in Part 2: Extended state that adds a call counter on top of the built-in 'messages' list
class CustomAgentState(AgentState):
    model_calls: int  # Tracks how many times the model node has been called in this run

# --- Lifecycle hook nodes ---
# These extra nodes don't change the core agent logic — they add logging/observability
# by running code AROUND the model invocation. Useful for debugging and monitoring.

def on_start(state: CustomAgentState):
    # Called once at the very beginning of each graph run
    print("Our graph execution started!")

def on_end(state: CustomAgentState):
    # Called once at the very end, just before the graph finishes
    print("Our graph execution just ended!")

def before_model_node(state: CustomAgentState):
    # Called immediately BEFORE each model invocation — increments and logs the call counter
    model_call_id = state.get('model_calls', 0) + 1
    print('=' * 20)
    print(f"Starting model call #{model_call_id}...")
    return { "model_calls": model_call_id }  # Update the counter in the state

def after_model_node(state: CustomAgentState):
    # Called immediately AFTER each model invocation — logs which call just finished
    model_call_id = state.get('model_calls', 0)
    print(f"Finished model call #{model_call_id}.")
    print('=' * 20)
    # No state update needed — this node only logs

def model_node(state: CustomAgentState):
    # The actual LLM call — prepends the system prompt to the full conversation history
    response = model.invoke([SystemMessage(SYSTEM_PROMPT), *state["messages"]])
    return { "messages": [response] }

In [ ]:
# Same routing logic as Part 1 — determines whether the model wants to call a tool
def has_pending_tool_calls(state: CustomAgentState) -> bool:
    messages = state.get("messages", [])
    if not messages:
        return False  # No messages yet — nothing to check

    last_message = messages[-1]
    # True if the model's last response includes one or more tool call requests
    return isinstance(last_message, AIMessage) and last_message.tool_calls

In [ ]:
# Build the extended agent graph with lifecycle hooks wrapping the model node.
#
# Full execution flow:
#   START -> on_start -> before_model -> model -> after_model
#         -> (tool call?) -> tools -> before_model  [loop back]
#         -> (done?)      -> on_end -> END
graph_builder = StateGraph(CustomAgentState)

# Register ALL nodes including the lifecycle hooks
graph_builder.add_node("on_start", on_start)
graph_builder.add_node("on_end", on_end)
graph_builder.add_node("before_model", before_model_node)
graph_builder.add_node("after_model", after_model_node)
graph_builder.add_node("model", model_node)
graph_builder.add_node("tools", ToolNode(TOOLS))

# Wire the execution order
graph_builder.add_edge(START, "on_start")
graph_builder.add_edge("on_start", "before_model")
graph_builder.add_edge("before_model", "model")
graph_builder.add_edge("model", "after_model")

# After the model finishes: go to tools if needed, otherwise wrap up
graph_builder.add_conditional_edges(
    "after_model",
    lambda x: "tools" if has_pending_tool_calls(x) else "on_end",
    ["tools", "on_end"]
)

# After tools run, go back to before_model to log the next call before re-invoking the model
graph_builder.add_edge("tools", "before_model")
graph_builder.add_edge("on_end", END)

graph = graph_builder.compile()

In [ ]:
# Visualize the extended agent graph — compare it to Part 1 to see the extra lifecycle nodes
display_graph(graph, Path("/content/graph.png"))

In [ ]:
# Run the agent with the same question as Part 1.
# Watch the lifecycle hook messages being printed as the graph executes —
# you'll see "Starting model call #1...", then the tool calls, then "Finished model call #1."
final_state = graph.invoke(
    input={
        "messages": [HumanMessage("What's the weather in Tokyo and what is LangGraph, briefly?")]
    }
)

In [ ]:
# Print the full conversation log
print_conversation(final_state["messages"])

In [ ]:
# Inspect the raw final state — note the 'model_calls' counter showing how many LLM calls were made
final_state